# Tool Calling 流式解析

最小实现：按 OpenAI Responses API 的流式事件，把 function call 的 arguments 增量拼起来并解析成最终结果。

In [ ]:
import json

def parse_tool_calls(events):
    # calls 字典以 output_index 为键，存储每个工具调用的中间状态
    # output_index 是模型输出的位置索引，用于区分同时触发的多个工具调用
    calls = {}

    for event in events:
        # 事件1：模型宣布"我要调用一个工具"，此时 arguments 还是空的
        if event["type"] == "response.output_item.added":
            item = event["item"]
            if item["type"] == "function_call":
                calls[event["output_index"]] = {
                    "type": "function_call",
                    "id": item["id"],           # 输出项 id
                    "call_id": item["call_id"], # 工具调用 id，后续将结果返回给模型时需要带上
                    "name": item["name"],       # 要调用的函数名
                    "arguments": ""             # arguments 还未到达，先置空
                }

        # 事件2：arguments 的增量片段（流式传输，可能会分多次到达）
        elif event["type"] == "response.function_call_arguments.delta":
            calls[event["output_index"]]["arguments"] += event["delta"]  # 逐片拼接

        # 事件3：arguments 传输完毕，用完整字符串覆盖（比自己拼接更可靠）
        elif event["type"] == "response.function_call_arguments.done":
            calls[event["output_index"]]["arguments"] = event["arguments"]

    # 按 output_index 排序保证顺序，然后将 arguments 从 JSON 字符串解析为字典
    return [
        {**call, "arguments": json.loads(call["arguments"])}
        for _, call in sorted(calls.items())
    ]


# ── 模拟流式事件序列 ──────────────────────────────────────────────
# 真实场景中这些 events 来自 OpenAI Responses API 的 SSE 流
events = [
    {
        # 第1个事件：模型声明要调用 get_weather，但 arguments 尚未传输
        "type": "response.output_item.added",
        "output_index": 0,
        "item": {
            "id": "fc_1",
            "type": "function_call",
            "call_id": "call_1",
            "name": "get_weather"
        }
    },
    {
        # 第2个事件：arguments 的第一个片段（流式传输，JSON 尚不完整）
        "type": "response.function_call_arguments.delta",
        "output_index": 0,
        "item_id": "fc_1",
        "delta": "{\"city\": \"Bei"
    },
    {
        # 第3个事件：arguments 的第二个片段，拼接后得到完整 JSON
        "type": "response.function_call_arguments.delta",
        "output_index": 0,
        "item_id": "fc_1",
        "delta": "jing\", \"unit\": \"celsius\"}"
    },
    {
        # 第4个事件：传输结束，携带完整 arguments 字符串
        "type": "response.function_call_arguments.done",
        "output_index": 0,
        "item_id": "fc_1",
        "arguments": "{\"city\": \"Beijing\", \"unit\": \"celsius\"}"
    }
]

parse_tool_calls(events)
